In [ ]:
import os
import cv2
import numpy as np
import time
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import tensorflow as tf
from tensorflow.keras.applications import ResNet50, ResNet152, ResNet101
from tensorflow.keras.applications.resnet import preprocess_input
from skimage import exposure
from skimage.io import imread
from skimage.transform import resize
from skimage.color import rgb2gray
import pandas as pd
from skimage.filters import gabor_kernel
from scipy import ndimage as ndi
from scipy.fftpack import dct
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Load pre-trained ResNet-50 model
resnet_50 = ResNet50(weights='imagenet', include_top=False, pooling='max')

# Define image directory and labels
image_dir = r"/kaggle/input/annocerv1555/Augmented and preprocessed"
labels = ['Class1', 'Class2', 'Class3']

# Create an empty dataframe to store the features and labels
res_50_df = pd.DataFrame(columns=['features', 'labels'])

# Function to extract features from an image and store them in the dataframe
def extract_features(image_path, label):
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=(224, 224))
    x = tf.keras.preprocessing.image.img_to_array(img)
    x = tf.expand_dims(x, axis=0)
    x = preprocess_input(x)

    features = resnet_50.predict(x, verbose=0)[0]  # Extract features
    res_50_df.loc[len(res_50_df)] = [features, label]  # Append features and label to the dataframe

# Iterate through the images and extract features
for label_idx, label in enumerate(labels):
    label_dir = os.path.join(image_dir, label)
    for subdir, dirs, files in os.walk(label_dir):
        for file in files:
            extract_features(os.path.join(subdir, file), label_idx)

# Display the dataframe
print(res_50_df)

In [ ]:
print("Resnet50 dataframe", res_50_df)

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, precision_score, f1_score

# Separate features and labels
X = res_50_df['features'].tolist()
y = res_50_df['labels'].tolist()

print("ResNet50 with 5-fold cross-validation and linear SVM")

num_splits = 5

accuracy_scores = []
specificity_scores = []
sensitivity_scores = []
precision_scores = []
f1_scores = []

kfold = KFold(n_splits=num_splits, shuffle=True)
svm = SVC(kernel='linear')

for train_index, test_index in kfold.split(X):
    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    y_train, y_test = np.array(y)[train_index], np.array(y)[test_index]

    svm.fit(X_train, y_train)
    y_pred = svm.predict(X_test)

    conf_matrix = confusion_matrix(y_test, y_pred)

    tp = conf_matrix[0, 0]
    fp = conf_matrix[1, 0] + conf_matrix[2, 0]
    fn = conf_matrix[0, 1] + conf_matrix[0, 2]
    tn = conf_matrix[1, 1] + conf_matrix[2, 2] + conf_matrix[1, 2] + conf_matrix[2, 1]

    specificity = tn / (tn + fp)
    sensitivity = tp / (tp + fn)
    accuracy = (tp + tn) / (tp + tn + fp + fn)

    precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

    specificity_scores.append(specificity)
    sensitivity_scores.append(sensitivity)
    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    f1_scores.append(f1)

average_accuracy = np.mean(accuracy_scores)
average_specificity = np.mean(specificity_scores)
average_sensitivity = np.mean(sensitivity_scores)
average_precision = np.mean(precision_scores)
average_f1 = np.mean(f1_scores)

print(f"Average Accuracy: {average_accuracy:.6f}")
print(f"Average Specificity: {average_specificity:.6f}")
print(f"Average Sensitivity: {average_sensitivity:.6f}")
print(f"Average Precision: {average_precision:.6f}")
print(f"Average F1 Score: {average_f1:.6f}")


In [ ]:
# Load pre-trained ResNet-101 model
resnet_101 = ResNet101(weights='imagenet', include_top=False, pooling='max')

# Define image directory and labels
image_dir = r"/kaggle/input/annocerv1555/Augmented and preprocessed"
labels = ['Class1', 'Class2', 'Class3']

# Create an empty dataframe to store the features and labels
res_101_df = pd.DataFrame(columns=['features', 'labels'])

# Function to extract features from an image and store them in the dataframe
def extract_features(image_path, label):
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=(224, 224))
    x = tf.keras.preprocessing.image.img_to_array(img)
    x = tf.expand_dims(x, axis=0)
    x = preprocess_input(x)

    features = resnet_101.predict(x, verbose = 0)[0]  # Extract features
    res_101_df.loc[len(res_101_df)] = [features, label]  # Append features and label to the dataframe

# Iterate through the images and extract features
for label_idx, label in enumerate(labels):
    label_dir = os.path.join(image_dir, label)
    for subdir, dirs, files in os.walk(label_dir):
        for file in files:
            extract_features(os.path.join(subdir, file), label_idx)

# Display the dataframe
print(res_101_df)

In [ ]:
print("Resnet101 dataframe", res_101_df)

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, precision_score, f1_score

# Separate features and labels
X = res_101_df['features'].tolist()
y = res_101_df['labels'].tolist()

print("ResNet101 with 5-fold cross-validation and linear SVM")

num_splits = 5


accuracy_scores = []
specificity_scores = []
sensitivity_scores = []
precision_scores = []
f1_scores = []

kfold = KFold(n_splits=num_splits, shuffle=True)
svm = SVC(kernel='linear')

for train_index, test_index in kfold.split(X):
    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    y_train, y_test = np.array(y)[train_index], np.array(y)[test_index]

    svm.fit(X_train, y_train)
    y_pred = svm.predict(X_test)

    conf_matrix = confusion_matrix(y_test, y_pred)

    tp = conf_matrix[0, 0]
    fp = conf_matrix[1, 0] + conf_matrix[2, 0]
    fn = conf_matrix[0, 1] + conf_matrix[0, 2]
    tn = conf_matrix[1, 1] + conf_matrix[2, 2] + conf_matrix[1, 2] + conf_matrix[2, 1]

    specificity = tn / (tn + fp)
    sensitivity = tp / (tp + fn)
    accuracy = (tp + tn) / (tp + tn + fp + fn)

    precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

    specificity_scores.append(specificity)
    sensitivity_scores.append(sensitivity)
    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    f1_scores.append(f1)

# Average scores
average_accuracy = np.mean(accuracy_scores)
average_specificity = np.mean(specificity_scores)
average_sensitivity = np.mean(sensitivity_scores)
average_precision = np.mean(precision_scores)
average_f1 = np.mean(f1_scores)

# Output
print(f"Average Accuracy: {average_accuracy:.6f}")
print(f"Average Specificity: {average_specificity:.6f}")
print(f"Average Sensitivity: {average_sensitivity:.6f}")
print(f"Average Precision: {average_precision:.6f}")
print(f"Average F1 Score: {average_f1:.6f}")


In [ ]:
# Load pre-trained ResNet-152 model
resnet_152 = ResNet152(weights='imagenet', include_top=False, pooling='max')

# Define image directory and labels
image_dir = r"/kaggle/input/annocerv1555/Augmented and preprocessed"
labels = ['Class1', 'Class2', 'Class3']

# Create an empty dataframe to store the features and labels
res_152_df = pd.DataFrame(columns=['features', 'labels'])

# Function to extract features from an image and store them in the dataframe
def extract_features(image_path, label):
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=(224, 224))
    x = tf.keras.preprocessing.image.img_to_array(img)
    x = tf.expand_dims(x, axis=0)
    x = preprocess_input(x)

    features = resnet_152.predict(x, verbose=0)[0]  # Extract features
    res_152_df.loc[len(res_152_df)] = [features, label]  # Append features and label to the dataframe

# Iterate through the images and extract features
for label_idx, label in enumerate(labels):
    label_dir = os.path.join(image_dir, label)
    for subdir, dirs, files in os.walk(label_dir):
        for file in files:
            extract_features(os.path.join(subdir, file), label_idx)

# Display the dataframe
print(res_152_df)

In [ ]:
print("Resnet152 dataframe", res_152_df)

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, precision_score, f1_score

# Separate features and labels
X = res_152_df['features'].tolist()
y = res_152_df['labels'].tolist()

print("ResNet152 with 5-fold cross-validation and linear SVM")

num_splits = 5


accuracy_scores = []
specificity_scores = []
sensitivity_scores = []
precision_scores = []
f1_scores = []

kfold = KFold(n_splits=num_splits, shuffle=True)
svm = SVC(kernel='linear')

for train_index, test_index in kfold.split(X):
    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    y_train, y_test = np.array(y)[train_index], np.array(y)[test_index]
    
    svm.fit(X_train, y_train)
    y_pred = svm.predict(X_test)
    
    conf_matrix = confusion_matrix(y_test, y_pred)
    
    tp = conf_matrix[0, 0]
    fp = conf_matrix[1, 0] + conf_matrix[2, 0]
    fn = conf_matrix[0, 1] + conf_matrix[0, 2]
    tn = conf_matrix[1, 1] + conf_matrix[2, 2] + conf_matrix[1, 2] + conf_matrix[2, 1]
    
    specificity = tn / (tn + fp)
    sensitivity = tp / (tp + fn)
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    
    precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    
    specificity_scores.append(specificity)
    sensitivity_scores.append(sensitivity)
    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    f1_scores.append(f1)

# Compute and display average metrics
print(f"\nAverage Accuracy   : {np.mean(accuracy_scores):.6f}")
print(f"Average Specificity: {np.mean(specificity_scores):.6f}")
print(f"Average Sensitivity: {np.mean(sensitivity_scores):.6f}")
print(f"Average Precision  : {np.mean(precision_scores):.6f}")
print(f"Average F1-Score   : {np.mean(f1_scores):.6f}")


# Feature Fusion

In [ ]:
merged_features_list = []
min_length = min(len(res_50_df), len(res_152_df),len(res_101_df))
for i in range(min_length):
    merged_features = np.concatenate([res_50_df.loc[i, 'features'], res_152_df.loc[i, 'features'], res_101_df.loc[i, 'features']])
    merged_features_list.append(merged_features)

# Create a new dataframe with merged features and labels
merged_labels_list = res_50_df['labels'][:min_length]  # Using 'label' column from dct_df
merged_data = {'features': merged_features_list, 'labels': merged_labels_list}
merged_df= pd.DataFrame(merged_data)




In [ ]:
print("Length of features in merged_df:", len(merged_df['features'].iloc[-1]))

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, precision_score, f1_score

# Separate features and labels
X = merged_df['features'].tolist()
y = merged_df['labels'].tolist()

print("R50+ R101+ R152 with 5-fold cross-validation and linear SVM")

num_splits = 5

accuracy_scores = []
specificity_scores = []
sensitivity_scores = []
precision_scores = []
f1_scores = []

kfold = KFold(n_splits=num_splits, shuffle=True)
svm = SVC(kernel='linear')

for train_index, test_index in kfold.split(X):
    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    y_train, y_test = np.array(y)[train_index], np.array(y)[test_index]

    svm.fit(X_train, y_train)
    y_pred = svm.predict(X_test)

    conf_matrix = confusion_matrix(y_test, y_pred)

    tp = conf_matrix[0, 0]
    fp = conf_matrix[1, 0] + conf_matrix[2, 0]
    fn = conf_matrix[0, 1] + conf_matrix[0, 2]
    tn = conf_matrix[1, 1] + conf_matrix[2, 2] + conf_matrix[1, 2] + conf_matrix[2, 1]

    specificity = tn / (tn + fp)
    sensitivity = tp / (tp + fn)
    accuracy = (tp + tn) / (tp + tn + fp + fn)

    precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

    specificity_scores.append(specificity)
    sensitivity_scores.append(sensitivity)
    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    f1_scores.append(f1)

# Average results
print(f"\nAverage Accuracy   : {np.mean(accuracy_scores):.6f}")
print(f"Average Specificity: {np.mean(specificity_scores):.6f}")
print(f"Average Sensitivity: {np.mean(sensitivity_scores):.6f}")
print(f"Average Precision  : {np.mean(precision_scores):.6f}")
print(f"Average F1-Score   : {np.mean(f1_scores):.6f}")
